# 05 · Mô hình định giá

- **Người phụ trách:** Đỗ Khánh Duy
- **Đầu vào:** `data/processed/cars_cleaned.csv` (notebook 02), `data/processed/text_flags.csv` (notebook 04)
- **Đầu ra:** `models/car_pricing_model.pkl`, dùng ở notebook 06
- **Liên quan proposal:** mục tiêu *Thẩm định giá độc lập*; tiêu chí R² > 0,85 trên tập kiểm tra

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

CLEAN_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"
TEXT_FLAGS_PATH = ROOT / "data" / "processed" / "text_flags.csv"
MODEL_PATH = ROOT / "models" / "car_pricing_model.pkl"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_STATE = 42
TARGET_COLUMN = "price_million"
R2_TARGET = 0.85

## 1. Bài toán hồi quy

Mỗi dòng dữ liệu tương ứng với một tin rao bán xe. Mô hình hồi quy dự đoán **giá niêm yết** từ các đặc trưng như tuổi xe, số kilomet đã đi, hãng và dòng xe. Biến mục tiêu là `price_million`, có đơn vị **triệu VNĐ**, được chuẩn hóa ở notebook 02 theo `price_million = price / 1_000_000`. Ví dụ, giá trị 450 tương ứng với 450 triệu đồng.

### Chỉ số đánh giá

| Chỉ số | Cách tính | Ý nghĩa thực tế | Tiêu chí |
|---|---|---|---|
| **R²** | $1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$ | Đo mức biến thiên giá được mô hình giải thích. R² = 0,85 tương ứng giải thích 85% biến thiên giá trên tập đánh giá. R² = 1 là dự đoán hoàn hảo; R² = 0 tương đương dự đoán giá trung bình của tập đánh giá; R² âm là kém hơn mốc này. | **R² > 0,85 trên tập test**; càng cao càng tốt. |
| **MAE** | $\frac{1}{n}\sum_i \lvert y_i - \hat{y}_i \rvert$ | Sai lệch tuyệt đối trung bình, tính bằng triệu VNĐ. MAE = 30 nghĩa là giá dự đoán lệch trung bình 30 triệu đồng mỗi xe. | Càng thấp càng tốt. |
| **RMSE** | $\sqrt{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}$ | Căn sai số bình phương trung bình, cũng tính bằng triệu VNĐ. Bình phương sai số làm các lần định giá sai lớn có trọng số cao hơn; RMSE lớn hơn nhiều so với MAE gợi ý có những sai lệch lớn cần phân tích. | Càng thấp càng tốt. |

Trong các công thức, $y_i$ là giá niêm yết thực tế, $\hat{y}_i$ là giá dự đoán, $\bar{y}$ là giá trung bình của tập đánh giá và $n$ là số tin được đánh giá. R² không phải tỷ lệ xe được dự đoán đúng hoặc phần trăm sai số giá.

Ngưỡng 0,85 là mục tiêu cần kiểm chứng sau huấn luyện. Chỉ kết luận đạt khi R² trên tập test **lớn hơn** ngưỡng này; báo cáo đồng thời MAE và RMSE để biết mức sai lệch bằng tiền. Việc chọn mô hình và tinh chỉnh sử dụng tập train cùng cross-validation, dành tập test cho đánh giá cuối cùng.

Không sử dụng `price`, `price_string` hoặc `price_million` làm đặc trưng đầu vào vì chúng trực tiếp chứa giá cần dự đoán. Phần 2 sẽ xác định tập đặc trưng; phần 3 và 4 sẽ chia dữ liệu và huấn luyện.

Dữ liệu đầu vào đã được notebook 02 điền ODO bằng trung vị và lọc ngoại lai IQR trước khi chia tập. Kết quả sau này phản ánh tập dữ liệu đã xử lý này; các bước tiền xử lý mới ở notebook 05 cần được fit trên tập train trong Pipeline.

Tham khảo cách tính: [scikit-learn: Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

In [2]:
df = pd.read_csv(CLEAN_PATH, low_memory=False)

if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Thiếu biến mục tiêu {TARGET_COLUMN} trong {CLEAN_PATH.name}.")

y = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
invalid_price_mask = ~np.isfinite(y) | y.le(0)

if invalid_price_mask.any():
    raise ValueError(
        f"Có {invalid_price_mask.sum():,} giá bị thiếu, không hữu hạn hoặc không dương. "
        "Cần kiểm tra dữ liệu trước khi huấn luyện."
    )
if y.size < 2 or y.nunique() < 2:
    raise ValueError("Cần ít nhất hai tin với giá khác nhau để đánh giá R².")

df[TARGET_COLUMN] = y

target_summary = pd.Series(
    {
        "Số tin": y.size,
        "Giá thấp nhất (triệu VNĐ)": y.min(),
        "Giá trung vị (triệu VNĐ)": y.median(),
        "Giá trung bình (triệu VNĐ)": y.mean(),
        "Giá cao nhất (triệu VNĐ)": y.max(),
    },
    name=TARGET_COLUMN,
).to_frame()

target_summary

,price_million
Số tin,13520.000000
Giá thấp nhất (triệu VNĐ),20.000000
Giá trung vị (triệu VNĐ),425.000000
Giá trung bình (triệu VNĐ),456.457285
Giá cao nhất (triệu VNĐ),1180.000000


### Hàm đánh giá dùng ở bước huấn luyện

`evaluate_regression(y_true, y_pred)` nhận giá thực tế và giá dự đoán theo cùng thứ tự tin, cùng đơn vị triệu VNĐ. Hàm trả về R², MAE, RMSE và kết quả kiểm tra ngưỡng R². Ví dụ ở phần 4: `test_metrics = evaluate_regression(y_test, model.predict(X_test))`.

Phần 1 chuẩn bị biến mục tiêu và cách đánh giá; chưa có kết quả của mô hình được huấn luyện.

In [3]:
from car_pricing.evaluation import evaluate_regression

## 2. Chọn đặc trưng và mã hóa

Đặc trưng được chọn theo danh sách cố định trong `car_pricing/features.py`. Mỗi tin có **21 đặc trưng** thuộc ba nhóm:

| Nhóm | Đặc trưng | Cách xử lý |
|---|---|---|
| **Số** | `car_age`, `mileage_v2` | Điền giá trị thiếu bằng trung vị của train, sau đó chuẩn hóa bằng StandardScaler. |
| **Danh mục** | `carbrand_name`, `carmodel_name`, `gearbox`, `fuel`, `cartype`, `carcolor`, `carorigin`, `carseats`, `region_name` | Gán nhãn `Không rõ` cho giá trị thiếu và one-hot encoding; gom nhóm hiếm riêng cho dòng xe. |
| **Cờ văn bản** | `co_mo_ta` và 9 cờ `vb_*` từ notebook 04 | Giữ nguyên giá trị 0/1, không chuẩn hóa. |

Tuổi xe biểu diễn thông tin năm sản xuất nên không dùng thêm `mfdate`. Các cột giá, mã tin, mã danh mục trùng thông tin nhãn, tiêu đề, mô tả thô và thời gian thu thập nằm ngoài tập đầu vào.

Ghép `text_flags.csv` với dữ liệu sạch theo `list_id`, kiểm tra khóa duy nhất, không thiếu và khớp giữa hai file. Phép ghép giữ thứ tự tin để `X` và `y` tương ứng từng dòng; cờ thiếu hoặc khác 0/1 sẽ báo lỗi để kiểm tra notebook 04.

`co_mo_ta` đánh dấu có mô tả được thu thập. Notebook 04 tìm từ khóa trên cả tiêu đề và mô tả, vì vậy `co_mo_ta = 0` vẫn có thể đi kèm cờ `vb_* = 1`. Các cờ phản ánh nội dung người bán viết.

In [4]:
from IPython.display import display

from car_pricing.features import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    TEXT_FEATURES,
    build_feature_dataset,
)
from car_pricing.preprocessing import build_preprocessor

text_flags = pd.read_csv(TEXT_FLAGS_PATH, low_memory=False)
modeling_data = build_feature_dataset(df, text_flags)
X = modeling_data.loc[:, list(FEATURE_COLUMNS)].copy()
y = modeling_data[TARGET_COLUMN].copy()

feature_groups = {
    "Số": NUMERIC_FEATURES,
    "Danh mục": CATEGORICAL_FEATURES,
    "Cờ văn bản": TEXT_FEATURES,
}
feature_summary = pd.DataFrame(
    [
        {
            "Nhóm": group,
            "Đặc trưng": column,
            "Giá trị thiếu": int(X[column].isna().sum()),
            "Tỷ lệ thiếu (%)": round(X[column].isna().mean() * 100, 2),
            "Số giá trị": X[column].nunique(),
        }
        for group, columns in feature_groups.items()
        for column in columns
    ]
)

print(f"Đã ghép {len(modeling_data):,} tin; X có {X.shape[1]} đặc trưng; y là {y.name}.")
display(feature_summary)

Đã ghép 13,520 tin; X có 21 đặc trưng; y là price_million.


,Nhóm,Đặc trưng,Giá trị thiếu,Tỷ lệ thiếu (%),Số giá trị
0,Số,car_age,0,0.00,43
1,Số,mileage_v2,0,0.00,1862
2,Danh mục,carbrand_name,0,0.00,56
3,Danh mục,carmodel_name,0,0.00,372
4,Danh mục,gearbox,0,0.00,3
5,Danh mục,fuel,0,0.00,4
6,Danh mục,cartype,5096,37.69,9
7,Danh mục,carcolor,521,3.85,12
8,Danh mục,carorigin,6381,47.20,10
9,Danh mục,carseats,3489,25.81,10


### Gom nhóm dòng xe hiếm

CSV hiện có **372** giá trị `carmodel_name`; **211** dòng xe có dưới 10 tin trên toàn bộ dữ liệu. Bảng dưới kiểm tra lại số liệu mỗi lần chạy, thay cho con số 422 trong hướng dẫn ban đầu.

Chọn `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")`: các dòng xe có dưới 10 tin trong **tập train** dùng chung một cột nhóm hiếm. Dòng xe mới ở test được đưa vào nhóm hiếm nếu nhóm này tồn tại; nếu train không có nhóm hiếm, encoder trả về các cột dòng xe bằng 0. Cách này giảm số cột và không dùng giá mục tiêu để mã hóa. Target encoding cần cơ chế cross-fitting để kiểm soát rò rỉ mục tiêu.

Thống kê trên toàn bộ dữ liệu ở đây phục vụ mô tả. Việc xác định nhóm hiếm thực tế chỉ diễn ra khi fit Pipeline trên train.

Tham khảo: [scikit-learn: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [5]:
MIN_MODEL_FREQUENCY = 10

model_counts = X["carmodel_name"].value_counts()
rare_models = model_counts[model_counts < MIN_MODEL_FREQUENCY]
model_summary = pd.DataFrame(
    [
        {
            "Số dòng xe": model_counts.size,
            "Dòng xe dưới ngưỡng": rare_models.size,
            "Số tin thuộc dòng hiếm": int(rare_models.sum()),
            "Ngưỡng số tin": MIN_MODEL_FREQUENCY,
        }
    ]
)

model_summary

,Số dòng xe,Dòng xe dưới ngưỡng,Số tin thuộc dòng hiếm,Ngưỡng số tin
0,372,211,630,10


### Chuẩn bị bộ tiền xử lý

`car_pricing/preprocessing.py` tạo `ColumnTransformer` với Pipeline riêng cho số, danh mục và dòng xe; cờ văn bản được chuyển tiếp trực tiếp. Encoder cho các danh mục còn lại dùng `handle_unknown="ignore"` để nhận giá trị mới khi dự đoán.

Ở phần này chỉ khởi tạo `preprocessor`, **chưa gọi fit hoặc fit_transform**. Phần 3 chia train/test và tạo Pipeline gồm tiền xử lý cùng mô hình; phần 4 fit Pipeline trên `X_train, y_train`. Trung vị, tham số chuẩn hóa, các nhóm hiếm và cột one-hot đều được học từ train; test chỉ dùng transform hoặc predict.

Các bước mới này kiểm soát việc học từ test tại notebook 05. Việc điền ODO và lọc IQR trước đó trong notebook 02 đã được nêu ở phần 1.

In [6]:
preprocessor = build_preprocessor(min_model_frequency=MIN_MODEL_FREQUENCY)

encoding_summary = pd.DataFrame(
    [
        {"Nhóm": "Số", "Số cột đầu vào": len(NUMERIC_FEATURES), "Xử lý": "Median imputation + StandardScaler"},
        {"Nhóm": "Danh mục", "Số cột đầu vào": len(CATEGORICAL_FEATURES) - 1, "Xử lý": "Nhãn thiếu + OneHotEncoder"},
        {"Nhóm": "Dòng xe", "Số cột đầu vào": 1, "Xử lý": f"Nhãn thiếu + OneHotEncoder, min_frequency={MIN_MODEL_FREQUENCY}"},
        {"Nhóm": "Cờ văn bản", "Số cột đầu vào": len(TEXT_FEATURES), "Xử lý": "Giữ nguyên 0/1"},
    ]
)

encoding_summary

,Nhóm,Số cột đầu vào,Xử lý
0,Số,2,Median imputation + StandardScaler
1,Danh mục,8,Nhãn thiếu + OneHotEncoder
2,Dòng xe,1,"Nhãn thiếu + OneHotEncoder, min_frequency=10"
3,Cờ văn bản,10,Giữ nguyên 0/1


## 3. Chia tập train/test

Dùng `train_test_split` để chia **80% train / 20% test**, với `random_state=42` và xáo trộn trước khi chia. Seed cố định giúp tái lập cách chia khi dữ liệu và thứ tự tin đầu vào giữ nguyên.

Chia đồng thời `X`, `y` và `list_id` để đặc trưng, giá và mã tin vẫn tương ứng từng dòng. `list_id` chỉ dùng kiểm tra hai tập không trùng tin, không đưa vào đặc trưng mô hình. Giữ các đặc trưng ở dạng chưa mã hóa sau khi chia.

Tập train dùng để học tiền xử lý, huấn luyện và cross-validation khi chọn mô hình hoặc tinh chỉnh. Tập test được giữ lại để đánh giá cuối cùng bằng R², MAE và RMSE; không dùng test để fit hoặc điều chỉnh mô hình.

Bảng dưới ghi số tin, tỷ lệ, số đặc trưng và giá trung vị của mỗi tập.

Tham khảo: [scikit-learn: train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [7]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2

if not X.index.equals(y.index) or not X.index.equals(modeling_data.index):
    raise ValueError("Đặc trưng, giá và mã tin phải có cùng thứ tự dòng trước khi chia tập.")

X_train, X_test, y_train, y_test, train_listing_ids, test_listing_ids = train_test_split(
    X,
    y,
    modeling_data["list_id"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

if not X_train.index.equals(y_train.index) or not X_test.index.equals(y_test.index):
    raise ValueError("Đặc trưng và giá không khớp dòng sau khi chia tập.")
if train_listing_ids.isin(test_listing_ids).any():
    raise ValueError("Có tin xuất hiện đồng thời trong train và test.")

split_summary = pd.DataFrame(
    [
        {
            "Tập": name,
            "Số tin": len(features),
            "Tỷ lệ (%)": round(len(features) / len(X) * 100, 2),
            "Số đặc trưng": features.shape[1],
            "Giá trung vị (triệu VNĐ)": target.median(),
        }
        for name, features, target in (
            ("Train", X_train, y_train),
            ("Test", X_test, y_test),
        )
    ]
)

split_summary

,Tập,Số tin,Tỷ lệ (%),Số đặc trưng,Giá trung vị (triệu VNĐ)
0,Train,10816,80.0,21,425.0
1,Test,2704,20.0,21,426.0


### Đặt tiền xử lý trong Pipeline

`build_model_pipeline` trong `car_pricing/preprocessing.py` tạo một Pipeline gồm bộ tiền xử lý và mô hình. Mỗi lần gọi tạo preprocessor mới và clone mô hình, để các lần huấn luyện có trạng thái học độc lập.

Pipeline chứa toàn bộ bước điền giá trị thiếu, chuẩn hóa số, gom dòng xe hiếm và one-hot encoding đã định nghĩa ở phần 2. Khi phần 4 fit Pipeline trên `X_train, y_train`, các bước này chỉ học từ train. Lệnh `baseline_pipeline.predict(X_test)` sử dụng lại các tham số đã học rồi dự đoán trên test.

Cell dưới chuẩn bị `baseline_pipeline` với Linear Regression, **chưa gọi fit**. Các mô hình khác ở phần 4 dùng cùng hàm tạo Pipeline; cross-validation cũng phải nhận Pipeline hoàn chỉnh cùng đặc trưng chưa mã hóa.

Phạm vi kiểm soát rò rỉ ở đây là các bước mới trong notebook 05. Giới hạn của dữ liệu đã xử lý ở notebook 02 được nêu trong phần 1.

Tham khảo: [scikit-learn: tránh rò rỉ dữ liệu với Pipeline](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [8]:
from sklearn.linear_model import LinearRegression

from car_pricing.preprocessing import build_model_pipeline

baseline_pipeline = build_model_pipeline(
    LinearRegression(),
    min_model_frequency=MIN_MODEL_FREQUENCY,
)

pipeline_summary = pd.DataFrame(
    [
        {"Bước": "preprocessor", "Thành phần": "ColumnTransformer", "Trạng thái": "Chưa fit"},
        {"Bước": "model", "Thành phần": "LinearRegression", "Trạng thái": "Chưa fit"},
    ]
)

pipeline_summary

,Bước,Thành phần,Trạng thái
0,preprocessor,ColumnTransformer,Chưa fit
1,model,LinearRegression,Chưa fit


## 4. Huấn luyện và so sánh mô hình

Huấn luyện bốn mô hình trên cùng `X_train, y_train` đã chia ở phần 3. Mỗi mô hình có Pipeline riêng chứa toàn bộ tiền xử lý; tập test chỉ đi qua predict để tính chỉ số đánh giá.

| Mô hình | Vai trò | Cấu hình khởi đầu |
|---|---|---|
| **Linear Regression** | Baseline tuyến tính để đối chiếu các mô hình cây. | Tham số mặc định của scikit-learn. |
| **Decision Tree** | Học quan hệ phi tuyến bằng một cây quyết định. | `min_samples_leaf=2`, `random_state=42`. |
| **Random Forest** | Tổng hợp nhiều cây học trên các mẫu bootstrap. | `n_estimators=300`, `min_samples_leaf=2`, `random_state=42`. |
| **XGBoost** | Boosting cây theo sai số của các bước trước. | `n_estimators=400`, `max_depth=6`, `learning_rate=0.05`, `subsample=0.9`, `colsample_bytree=0.9`, `reg_lambda=1.0`, `random_state=42`. |

Cấu hình được cố định trước khi xem kết quả test. Random Forest và XGBoost dùng 4 luồng CPU; XGBoost dùng `tree_method="hist"` và `objective="reg:squarederror"`.

Pipeline XGBoost xuất ma trận **dense** để giữ các giá trị 0 của one-hot, cờ văn bản và số đã chuẩn hóa. Các Pipeline còn lại giữ đầu ra sparse. Bộ cây của XGBoost coi phần tử không được lưu trong sparse là giá trị thiếu; định dạng dense giúp giá trị 0 được biểu diễn rõ ràng. Tham khảo: [XGBoost: sparse và dense](https://xgboost.readthedocs.io/en/stable/faq.html#why-do-i-see-different-results-with-sparse-and-dense-data).

`car_pricing/training.py` quản lý cấu hình và vòng lặp huấn luyện; `car_pricing/evaluation.py` tính các chỉ số đã định nghĩa ở phần 1. Kết quả gồm `trained_models` (Pipeline đã fit), `model_predictions` (giá dự đoán theo index từng tập) và `comparison_table` (chỉ số chưa làm tròn).

In [9]:
from car_pricing.training import build_model_pipelines, train_and_compare_models

N_JOBS = 4

model_pipelines = build_model_pipelines(
    random_state=RANDOM_STATE,
    min_model_frequency=MIN_MODEL_FREQUENCY,
    n_jobs=N_JOBS,
)
trained_models, model_predictions, comparison_table = train_and_compare_models(
    model_pipelines,
    X_train,
    y_train,
    X_test,
    y_test,
    r2_target=R2_TARGET,
)
baseline_pipeline = trained_models["Linear Regression"]

print(f"Đã huấn luyện {len(trained_models)} mô hình trên {len(X_train):,} tin train.")

Đã huấn luyện 4 mô hình trên 10,816 tin train.


### Bảng đánh giá trên train và test

Báo cáo **R², MAE và RMSE trên cả hai tập**. MAE và RMSE có đơn vị **triệu VNĐ**; R² càng cao và hai sai số càng thấp thì kết quả càng tốt. Cột `R2_gap` là R² train trừ R² test: chênh lệch lớn gợi ý mô hình học tốt trên train nhưng khái quát kém hơn trên test.

`Meets_R2_target` kiểm tra điều kiện **R² test > 0,85** trên điểm chưa làm tròn. Bảng hiển thị làm tròn R² đến 4 chữ số và MAE/RMSE đến 2 chữ số; `comparison_table` giữ nguyên độ chính xác cho các bước sau.

Đây là kết quả của cấu hình khởi đầu trên dữ liệu đã xử lý ở notebook 02. Phần 5 cần dùng cross-validation trên train để tinh chỉnh và chọn mô hình; bảng test ở đây phục vụ báo cáo so sánh.

In [10]:
comparison_display = comparison_table.round(
    {
        "R2_train": 4,
        "MAE_train": 2,
        "RMSE_train": 2,
        "R2_test": 4,
        "MAE_test": 2,
        "RMSE_test": 2,
        "R2_gap": 4,
    }
)
comparison_display["Meets_R2_target"] = comparison_display["Meets_R2_target"].map(
    {True: "Đạt", False: "Chưa đạt"}
)

comparison_display

,Model,R2_train,MAE_train,RMSE_train,R2_test,MAE_test,RMSE_test,R2_gap,Meets_R2_target
0,Linear Regression,0.8318,66.14,101.18,0.7996,70.94,108.19,0.0322,Chưa đạt
1,Decision Tree,0.9734,19.53,40.25,0.7959,63.02,109.19,0.1775,Chưa đạt
2,Random Forest,0.9661,24.62,45.45,0.8767,49.74,84.85,0.0893,Đạt
3,XGBoost,0.9257,44.19,67.27,0.8723,55.02,86.36,0.0534,Đạt


## 5. Tinh chỉnh siêu tham số

> **Hướng dẫn (xoá khi viết xong):** áp dụng cho 1–2 mô hình tốt nhất; ghi lại các tham số đã thử và kết quả.

## 6. Phân tích sai số

> **Hướng dẫn (xoá khi viết xong):** residual plot, feature importance; phân khúc nào dự đoán sai nhiều nhất và vì sao.

## 7. Lưu mô hình

> **Hướng dẫn (xoá khi viết xong):** lưu **cả pipeline** (mã hóa + mô hình), không chỉ mô hình, để notebook 06 đưa thẳng thông số thô vào dự đoán. File phải dưới 100 MB (giới hạn của GitHub).